## 04_train_xgboost  (OLD workspace) — Practice model artifact 4 of 5

Old-school requirements bootstrap (lift-and-shift: same shape as production):
  setup_wd_path(dbutils)  -> PROJECT_PATH (absolute project root)
  prepare_requirements()  -> REQ_PATH (space-free /tmp staging; built-in
                             fallback when the file is not readable, e.g.
                             DBR 12.2 has no /Workspace FUSE)
  %pip install -r "$REQ_PATH" -q   <- unchanged production line
Trains on the table chosen in 00_setup_sample_data (practice_customers).


In [ ]:

%run ./lib/config_lib


In [ ]:

############################
# set working dir path + stage requirements
############################
import os

PROJECT_PATH = setup_wd_path(dbutils)
REQ_PATH = prepare_requirements(PROJECT_PATH)
print(f"PROJECT_PATH : {PROJECT_PATH}")
print(f"REQ_PATH     : {REQ_PATH}")


In [ ]:

%pip install -r "$REQ_PATH" -q


In [ ]:
dbutils.widgets.text("SCHEMA", "practice", "Schema for practice tables (OLD schema name — unchanged by migration)")

CFG = setup_config(
    config_path="",
    run_date="all",
    schema=dbutils.widgets.get("SCHEMA").strip(),
)

TRAIN_TABLE = CFG.table("practice_customers")
print(f"Training table: {TRAIN_TABLE}")
print(f"features: {FEATURES} | label: {LABEL}")


In [ ]:

import numpy as np
import xgboost
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

print(f"xgboost version: {xgboost.__version__}")

X, y, _ids = load_training_data(TRAIN_TABLE)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"rows: {len(y)} | train: {len(y_train)} | test: {len(y_test)}")

model = xgboost.XGBRegressor(
    n_estimators=50, max_depth=4, learning_rate=0.1, random_state=42
).fit(X_train, y_train)
rmse = float(np.sqrt(mean_squared_error(y_test, model.predict(X_test))))
print(f"RMSE: {rmse:.4f}")


In [ ]:

import mlflow

REG_NAME = CFG.model("practice_xgboost")  # Free Edition UC registry: <current_catalog>.default.<name>
with mlflow.start_run() as run:
    mlflow.log_param("model_type", "xgboost")
    mlflow.log_param("n_estimators", 50)
    mlflow.log_param("train_table", TRAIN_TABLE)
    mlflow.log_metric("rmse", rmse)
    mlflow.xgboost.log_model(model, artifact_path="model")
    mv = mlflow.register_model(f"runs:/{run.info.run_id}/model", REG_NAME)

print(f"REGISTERED  name='{mv.name}'  version={mv.version}")
